# Bias Check

Inference only — no training. For each hidden cell fraction of the
`increasing-hidden-fraction` grid search we evaluate the van Rossum loss at two
settings of the scaling factors:

- **learnt** — the scaling factors at the lowest loss reached during that training run
- **correct** — the target scaling factors, which exactly undo the weight perturbation
  (1.0 everywhere once normalised)

If the correct scaling factors sit at a *higher* loss than the learnt ones, the loss
minimum is biased away from the truth: training converges to the wrong place because
the objective's minimum is in the wrong place, not because optimisation fails.

This is the pilot for a full sweep over the scaling factors — it evaluates the two
end points only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.visualization import (
    plot_r2_vs_parameter,
    use_project_style,
)
from connectome_snns.visualization.scaling_factors import (
    SF_PATHWAYS,
    plot_sf_vs_parameter,
)

use_project_style()

In [ ]:
config = load_experiment_config("experiment.toml")
RESULTS_DIR = config["output_dir"]

SF_SOURCE_LABELS = {"learnt": "Learnt minimum", "correct": "Correct (target)"}

## Results

In [ ]:
results = pd.read_csv(RESULTS_DIR / "inference_losses.csv")
fractions = sorted(results["hidden_cell_fraction"].unique())

losses = {
    SF_SOURCE_LABELS[source]: [
        float(
            results.loc[
                (results["hidden_cell_fraction"] == frac)
                & (results["sf_source"] == source),
                "van_rossum_loss",
            ].iloc[0]
        )
        for frac in fractions
    ]
    for source in SF_SOURCE_LABELS
}

results

## Loss at the Learnt Minimum vs the Correct Scaling Factors

Left: the inference loss for both settings. Right: the gap between them. A positive
gap means the correct scaling factors are *worse* under the loss than the learnt ones
— i.e. the objective is biased and the learnt minimum is not the true one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_r2_vs_parameter(
    fractions,
    losses,
    xlabel="Hidden Cell Fraction",
    ylabel="Van Rossum Loss",
    title="Inference Loss",
    ax=axes[0],
)

gap = np.array(losses[SF_SOURCE_LABELS["correct"]]) - np.array(
    losses[SF_SOURCE_LABELS["learnt"]]
)
axes[1].plot(fractions, gap, "o-", linewidth=1.5, markersize=4, color="C2")
axes[1].axhline(y=0.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[1].set_xlabel("Hidden Cell Fraction")
axes[1].set_ylabel("Loss(correct) \u2212 Loss(learnt)")
axes[1].set_title("Bias of the Loss Minimum", fontweight="bold")

fig.suptitle(
    "Learnt vs Correct Scaling Factors (inference only)",
    fontsize=14,
    fontweight="bold",
)
fig.tight_layout()
plt.show()

## Inference Loss vs the Loss Reached During Training

Sanity check that the inference pipeline reproduces the training objective: the
"learnt" inference loss should sit close to the minimum logged during training. Any
offset comes from evaluating over a different chunk window than the running training
loss did.

In [ ]:
learnt = results[results["sf_source"] == "learnt"].sort_values("hidden_cell_fraction")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    learnt["hidden_cell_fraction"],
    learnt["training_min_loss"],
    "o--",
    markersize=4,
    label="Training minimum (logged)",
)
ax.plot(
    learnt["hidden_cell_fraction"],
    learnt["van_rossum_loss"],
    "o-",
    markersize=4,
    label="Inference at learnt SFs",
)
ax.set_xlabel("Hidden Cell Fraction")
ax.set_ylabel("Van Rossum Loss")
ax.set_title("Inference vs Training Loss", fontweight="bold")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Where the Learnt Minimum Sits

The learnt scaling factors normalised by the correct ones — 1.0 is the truth, so the
distance from 1.0 is the parameter-space bias that the loss gap above corresponds to.

In [ ]:
sf_data = np.load(RESULTS_DIR / "scaling_factors.npz")
input_cell_type_names = list(sf_data["input_cell_type_names"])
output_cell_type_names = list(sf_data["output_cell_type_names"])

runs_by_fraction = {
    float(row["hidden_cell_fraction"]): row["run"] for _, row in learnt.iterrows()
}

normalised_sfs = {key: [] for key, _ in SF_PATHWAYS}
for frac in fractions:
    run = runs_by_fraction[frac]
    ratio = sf_data[f"{run}__learnt"] / sf_data[f"{run}__correct"]
    for src_idx, src_name in enumerate(input_cell_type_names):
        for tgt_idx, tgt_name in enumerate(output_cell_type_names):
            key = f"{src_name}_to_{tgt_name}"
            if key in normalised_sfs:
                normalised_sfs[key].append(float(ratio[src_idx, tgt_idx]))

plot_sf_vs_parameter(
    fractions,
    {key: np.array(values) for key, values in normalised_sfs.items()},
    xlabel="Hidden Cell Fraction",
    suptitle="Learnt Scaling Factors at the Minimum (normalised, target = 1)",
)
plt.show()

## Firing Rates

Mean visible firing rate under each setting, against the teacher. Systematic
over- or under-shoot at the correct scaling factors points at where the bias comes
from.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for ax, cell_type in zip(axes, ["excitatory", "inhibitory"]):
    for source, label in SF_SOURCE_LABELS.items():
        subset = results[results["sf_source"] == source].sort_values(
            "hidden_cell_fraction"
        )
        ax.plot(
            subset["hidden_cell_fraction"],
            subset[f"firing_rate/student_visible_{cell_type}_mean"],
            "o-",
            markersize=4,
            label=label,
        )
    ax.plot(
        subset["hidden_cell_fraction"],
        subset[f"firing_rate/teacher_visible_{cell_type}_mean"],
        "k--",
        linewidth=1.5,
        label="Teacher",
    )
    ax.set_xlabel("Hidden Cell Fraction")
    ax.set_ylabel("Mean Firing Rate (Hz)")
    ax.set_title(cell_type.capitalize())
    ax.set_ylim(0, None)
    ax.legend(fontsize=8)

fig.suptitle("Visible Firing Rates", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Summary

In [ ]:
summary = pd.DataFrame(
    {
        "Hidden Frac.": fractions,
        "Loss (learnt)": losses[SF_SOURCE_LABELS["learnt"]],
        "Loss (correct)": losses[SF_SOURCE_LABELS["correct"]],
        "Gap": gap,
        "Correct is worse": gap > 0,
    }
)
print(summary.to_string(index=False))